In [1]:
# Needed for databricks...
%pip install typing_extensions git+https://github.com/AgDMALabs-Public/ag-vision-dataops.git

In [ ]:
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

from ag_vision.pipelines import image_processing as ip

spark = SparkSession.builder.getOrCreate()

In [ ]:
# Only for Databricks
dbutils.widgets.text("REGEN", "False")

In [ ]:
# Set to True if you want to generate the whole table from scratch. (this will read all metadatafile)
# Set to False if you only want to add new data. (this will read only metadata files for images not in the table.)
REGEN = dbutils.widgets.get("REGEN")
REGEN = REGEN == "True"
TABLE_NAME = "use1_prod_artemis_catalog_3718194974443840.production.images_draft"

print(
    f"Regen is {REGEN}, if true it will recreate the whole table, if False it will append new data to the existing table.")

In [ ]:
catalog_name = "use1_prod_artemis_catalog_3718194974443840"
schema_name = 'production'
volume_name = 'data'

In [ ]:
# Load the root directory
root_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

df = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load(root_path) \
    .select("path")

img_paths_df = df.select("path").filter(
    # Ensure it's an image
    F.col("path").rlike(r"\.(jpg|jpeg|png|webp)$") &
    # Ensure 'im' folder is in the path
    F.col("path").contains("/im") &
    # Ensure the depth matches your 13-slash pattern, this will make sure annotation images don't make it in.
    (F.size(F.split(F.col("path"), "/")) >= 13)
)

# Collect
img_paths_list = [row.path for row in img_paths_df.toLocalIterator()]
print(f"Found {len(img_paths_list)} paths at the correct depth.")

In [ ]:
old_df = spark.table(TABLE_NAME).select("file_path").toPandas()
print(f"The Old Table len is {len(old_df)}")

In [ ]:
img_paths_list = [x.replace('dbfs:', '') for x in img_paths_list]

if not REGEN:
    s = pd.Series(img_paths_list)
    run_list = s[~s.isin(old_df['file_path'])].tolist()
else:
    run_list = img_paths_list

In [ ]:
print(f"The len of the run list is. {len(run_list)}")

In [ ]:
# This is used to generate the schema before we run the large spark cmd next.
if len(run_list) > 0:
    sample_df = ip.generate_images_table(img_list=run_list[:1],
                                         platform='db',
                                         project_index=6)

    sample_df['blur'] = sample_df['blur'].astype(int)

    my_schema = spark.createDataFrame(sample_df).schema.simpleString()

    df = spark.createDataFrame([(i,) for i in run_list], ["item"])

In [ ]:
def process_batch(iterator):
    """
    Processes an iterator of pandas DataFrames (chunks) and yields processed results.
    """
    for pdf in iterator:
        # pdf is a pandas DataFrame chunk
        items = pdf["item"].tolist()

        if not items:
            # Yield an empty DataFrame with the correct columns if the chunk is empty
            yield pd.DataFrame(columns=sample_df.columns)
        else:
            # Process the items and yield the resulting DataFrame
            yield ip.generate_images_table(img_list=items,
                                           platform='db',
                                           project_index=6)


In [ ]:
# generateing the table to all the entries in the run list with batch process
if len(run_list) > 0:
    img_spark_df = df.mapInPandas(process_batch,
                                  schema=my_schema)

In [ ]:
files_sdf = spark.createDataFrame([(path,) for path in img_paths_list], ["file_path"])

In [ ]:
# 1. Check if the table exists and REGEN is False
table_exists = spark.catalog.tableExists(TABLE_NAME)

if not REGEN and table_exists:
    # Create a DataFrame from img_paths_list to avoid driver memory issues with .isin()
    old_spark_df = spark.read.table(TABLE_NAME).join(files_sdf, "file_path", "inner")

    if len(run_list) > 0:
        final_spark_df = old_spark_df.unionByName(img_spark_df)
    else:
        final_spark_df = old_spark_df
else:
    # If REGEN is True or table doesn't exist, just use the new data
    final_spark_df = img_spark_df

#2. Write the final result back to the table
final_spark_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)